# City of Melbourne Generated-name Exploration

## Purpose

This notebook explores whether official City of Melbourne Open Data can provide
names for the Melbourne locations that received generated names during
Iteration 1.

The exploration is read-only. It does not modify the application-ready CSV or
apply any candidate names.

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests


def find_project_root(start=None):
    """Find the repository containing data and pipeline folders."""

    current = (
        Path.cwd()
        if start is None
        else Path(start).resolve()
    )

    for candidate in [current, *current.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "pipeline").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not find the project root."
    )


PROJECT_ROOT = find_project_root()

PLACES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vicmap"
    / "vicmap_app_ready.csv"
)

places = pd.read_csv(
    PLACES_PATH,
    encoding="utf-8-sig",
)

print("Rows:", len(places))
print("Columns:", len(places.columns))

Rows: 3237
Columns: 14


In [2]:
# Select Melbourne locations with Iteration 1 generated names.
melbourne_mask = (
    places["lga_name"].eq("MELBOURNE")
    & places["name_source"].eq(
        "generated_from_subtype"
    )
)

melbourne_unnamed = places.loc[
    melbourne_mask
].copy()

print(
    "Melbourne generated-name locations:",
    len(melbourne_unnamed),
)

display(
    melbourne_unnamed[
        [
            "place_id",
            "display_name",
            "activity_category",
            "feature_subtype",
            "longitude",
            "latitude",
        ]
    ].head()
)

Melbourne generated-name locations: 27


,place_id,display_name,activity_category,feature_subtype,longitude,latitude
78,vicmap_foi_1019251,Unnamed Basketball Court - Melbourne - 1019251,court,basketball court,144.925151,-37.788034
79,vicmap_foi_1206820,Unnamed Netball Court - Melbourne - 1206820,court,netball court,144.947382,-37.784832
276,vicmap_foi_1000946,Unnamed Park - Melbourne - 1000946,park_and_garden,park,144.954686,-37.824169
277,vicmap_foi_1002109,Unnamed Park - Melbourne - 1002109,park_and_garden,park,144.947508,-37.819905
278,vicmap_foi_643094,Unnamed Park - Melbourne - 643094,park_and_garden,park,144.936174,-37.790994


In [3]:
# Summarise the target records by category and subtype.
melbourne_target_summary = (
    melbourne_unnamed
    .groupby(
        ["activity_category", "feature_subtype"]
    )
    .size()
    .reset_index(name="locations")
)

display(melbourne_target_summary)

assert len(melbourne_unnamed) == 27
assert melbourne_unnamed["place_id"].is_unique
assert melbourne_unnamed[
    ["longitude", "latitude"]
].notna().all().all()

print("Melbourne target checks passed.")

,activity_category,feature_subtype,locations
0,court,basketball court,1
1,court,netball court,1
2,park_and_garden,park,5
3,playground,playground,8
4,sports_ground,sports ground,12


Melbourne target checks passed.


In [4]:
COM_API_BASE = (
    "https://data.melbourne.vic.gov.au/"
    "api/explore/v2.1/catalog/datasets"
)

COM_DATASETS = {
    "playgrounds": "playgrounds",
    "landmarks": (
        "landmarks-and-places-of-interest-"
        "including-schools-theatres-health-"
        "services-spor"
    ),
}


def fetch_com_geojson(dataset_id):
    """Retrieve one complete City of Melbourne dataset."""

    url = (
        f"{COM_API_BASE}/{dataset_id}/"
        "exports/geojson"
    )

    response = requests.get(
        url,
        timeout=60,
    )
    response.raise_for_status()

    payload = response.json()

    if payload.get("type") != "FeatureCollection":
        raise ValueError(
            f"{dataset_id} did not return GeoJSON."
        )

    return gpd.GeoDataFrame.from_features(
        payload["features"],
        crs="EPSG:4326",
    )

In [5]:
# Retrieve both official datasets in memory.
com_playgrounds = fetch_com_geojson(
    COM_DATASETS["playgrounds"]
)

com_landmarks = fetch_com_geojson(
    COM_DATASETS["landmarks"]
)

print("Playground records:", len(com_playgrounds))
print("Playground columns:")
print(com_playgrounds.columns.tolist())

print()

print("Landmark records:", len(com_landmarks))
print("Landmark columns:")
print(com_landmarks.columns.tolist())

Playground records: 45
Playground columns:
['geometry', 'name', 'council_re', 'features', 'location_d', 'geo_point_2d']

Landmark records: 242
Landmark columns:
['geometry', 'theme', 'sub_theme', 'feature_name']


In [6]:
display(com_playgrounds.head())
display(com_landmarks.head())

,geometry,name,council_re,features,location_d,geo_point_2d
0,"MULTIPOLYGON (((144.94716 -37.82193, 144.94716...",Docklands Park Playground,1450822,"Carousels, platforms, swing, slide, public toi...",NaN,"{'lon': 144.94722750144075, 'lat': -37.8218009..."
1,"MULTIPOLYGON (((144.97086 -37.802, 144.97085 -...",Carlton Gardens Playground,1450821,"Cubby with slide, swings, track glide, sandpit...",NaN,"{'lon': 144.97067458905988, 'lat': -37.8020962..."
2,"MULTIPOLYGON (((144.97408 -37.797, 144.97408 -...",Station Street Park Playground,1557352,Climbers x 3,NaN,"{'lon': 144.9739866829798, 'lat': -37.79728503..."
3,"MULTIPOLYGON (((144.94135 -37.79972, 144.9416 ...",North Melbourne Recreation Reserve Play Court,1618816,Play court space with basketball backboard only,NaN,"{'lon': 144.94146553497296, 'lat': -37.7998031..."
4,"MULTIPOLYGON (((144.92558 -37.79006, 144.92563...",Liddy Street Reserve Playground,1450827,"Combination unit, spring rider.",NaN,"{'lon': 144.9256728627734, 'lat': -37.79003673..."


,geometry,theme,sub_theme,feature_name
0,POINT (144.95138 -37.82114),Vacant Land,Current Construction Site - Commercial,Railway Good Shed No 2
1,POINT (144.97302 -37.81161),Transport,Railway Station,Parliament Railway Station
2,POINT (144.94145 -37.79883),Leisure/Recreation,Informal Outdoor Facility (Park/Garden/Reserve),North Melbourne Recreation Reserve
3,POINT (144.96112 -37.78702),Leisure/Recreation,Informal Outdoor Facility (Park/Garden/Reserve),Princes Park
4,POINT (144.96517 -37.82482),Office,Office,Donor Tissue Bank of Victoria


### Initial City of Melbourne source inspection

Both City of Melbourne datasets were successfully retrieved as GeoDataFrames in EPSG:4326.

The `Playgrounds` dataset contains named playground and play-court polygons. Its main fields include the facility `name`, a council reference identifier, a description of available `features`, polygon geometry and a representative geographic point. Because it provides detailed facility boundaries, it is the most suitable source for the Melbourne playground targets and may also support some court records.

The `Landmarks and Places of Interest` dataset contains named point locations with `theme`, `sub_theme` and `feature_name` fields. Relevant records can be filtered to the `Leisure/Recreation` theme, including informal outdoor facilities and major sports and recreation facilities. This source may support park, reserve, sports-ground and court records.

The two sources require different spatial matching methods. Target points can be matched directly to containing playground polygons, while landmark points require distance-based candidate matching. At this stage, the source data has only been inspected and no names have been applied to the Vicmap records.

### Source name and category profiling

Before spatial matching, the source name fields, geometry types and recreation
categories are examined. This step identifies usable named features and removes
unrelated landmark categories. No Vicmap names are modified.

In [8]:
def clean_source_name(series):
    """Trim names and convert blank values to missing."""

    cleaned = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    invalid_values = {
        "",
        "NO DATA",
        "N/A",
        "NA",
        "UNKNOWN",
        "UNNAMED",
    }

    return cleaned.mask(
        cleaned.str.upper().isin(invalid_values)
    )


# Clean names without changing source capitalisation.
com_playgrounds["name_clean"] = clean_source_name(
    com_playgrounds["name"]
)

com_landmarks["feature_name_clean"] = clean_source_name(
    com_landmarks["feature_name"]
)

In [9]:
# Summarise playground source quality.
playground_source_summary = pd.Series(
    {
        "total_records": len(com_playgrounds),
        "named_records": (
            com_playgrounds["name_clean"].notna().sum()
        ),
        "unique_names": (
            com_playgrounds["name_clean"].nunique()
        ),
        "missing_geometry": (
            com_playgrounds.geometry.isna().sum()
        ),
    },
    name="count",
)

# Summarise landmark source quality.
landmark_source_summary = pd.Series(
    {
        "total_records": len(com_landmarks),
        "named_records": (
            com_landmarks[
                "feature_name_clean"
            ].notna().sum()
        ),
        "unique_names": (
            com_landmarks[
                "feature_name_clean"
            ].nunique()
        ),
        "missing_geometry": (
            com_landmarks.geometry.isna().sum()
        ),
    },
    name="count",
)

display(playground_source_summary.to_frame())
display(landmark_source_summary.to_frame())

,count
total_records,45
named_records,45
unique_names,45
missing_geometry,0


,count
total_records,242
named_records,242
unique_names,233
missing_geometry,0


In [10]:
# Check which geometry types each source uses.
geometry_summary = pd.DataFrame(
    {
        "playgrounds": (
            com_playgrounds.geometry.geom_type
            .value_counts()
        ),
        "landmarks": (
            com_landmarks.geometry.geom_type
            .value_counts()
        ),
    }
).fillna(0).astype(int)

display(geometry_summary)

,playgrounds,landmarks
MultiPolygon,45,0
Point,0,242


In [11]:
# Inspect all landmark themes before filtering.
theme_summary = (
    com_landmarks
    .groupby("theme", dropna=False)
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

display(theme_summary)

,theme,records
4,Leisure/Recreation,63
7,Place Of Assembly,40
8,Place of Worship,31
13,Transport,26
0,Community Use,21
1,Education Centre,13
5,Mixed Use,11
2,Health Services,11
6,Office,11
9,Purpose Built,4


In [12]:
# Examine sub-themes under Leisure/Recreation.
recreation_landmarks = com_landmarks.loc[
    com_landmarks["theme"]
    .astype("string")
    .str.strip()
    .eq("Leisure/Recreation")
].copy()

recreation_subthemes = (
    recreation_landmarks
    .groupby("sub_theme", dropna=False)
    .agg(
        records=("feature_name_clean", "size"),
        named_records=(
            "feature_name_clean",
            "count",
        ),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

display(recreation_subthemes)

,sub_theme,records,named_records
2,Informal Outdoor Facility (Park/Garden/Reserve),37,37
3,Major Sports & Recreation Facility,14,14
5,"Outdoor Recreation Facility (Zoo, Golf Course)",4,4
1,Indoor Recreation Facility,3,3
4,Observation Tower/Wheel,2,2
6,Private Sports Club/Facility,2,2
0,Gymnasium/Health Club,1,1


In [13]:
# Confirm that both sources are usable.
assert com_playgrounds.crs is not None
assert com_landmarks.crs is not None

assert com_playgrounds.geometry.notna().all()
assert com_landmarks.geometry.notna().all()

assert com_playgrounds["name_clean"].notna().any()
assert recreation_landmarks[
    "feature_name_clean"
].notna().any()

print("City of Melbourne source profiling passed.")

City of Melbourne source profiling passed.


### Landmark category selection

The `Leisure/Recreation` theme contains 63 records across seven sub-themes, and
all 63 records have a usable feature name.

Two sub-themes directly correspond to the current Melbourne target categories:

- `Informal Outdoor Facility (Park/Garden/Reserve)`: 37 named records, suitable
  for `park_and_garden` targets and as contextual evidence for outdoor
  facilities.
- `Major Sports & Recreation Facility`: 14 named records, suitable for
  `sports_ground` and `court` targets.

`Private Sports Club/Facility` contains two named records and will be retained
only as a secondary review category because public access cannot be assumed.

The remaining sub-themes are not included in the initial matching. Indoor
facilities, observation attractions and gymnasiums do not correspond to the
current target feature types. Zoo and golf-course records are also excluded
because none of the 27 Melbourne targets has a matching subtype.

The initial landmark matching will therefore use the 51 records from the two
directly relevant sub-themes. The two private sports facilities may be examined
separately if the primary sources leave unmatched locations.

### Spatial matching with official playground polygons

The named City of Melbourne playground polygons are tested first because they
provide facility-level boundaries and names. A Vicmap target is treated as a
spatial candidate only when its point lies inside a named playground polygon.

This is a conservative containment test. It does not use nearest-neighbour
distance and does not modify any target names.

In [14]:
# Convert the 27 Melbourne targets to spatial points.
melbourne_points = gpd.GeoDataFrame(
    melbourne_unnamed.copy(),
    geometry=gpd.points_from_xy(
        melbourne_unnamed["longitude"],
        melbourne_unnamed["latitude"],
    ),
    crs="EPSG:4326",
)

print("Melbourne target points:", len(melbourne_points))

Melbourne target points: 27


In [15]:
# Keep named playground polygons and useful review fields.
named_playgrounds = com_playgrounds.loc[
    com_playgrounds["name_clean"].notna(),
    [
        "name_clean",
        "council_re",
        "features",
        "geometry",
    ],
].copy()

named_playgrounds = named_playgrounds.to_crs(
    "EPSG:4326"
)

print("Named playground polygons:", len(named_playgrounds))

Named playground polygons: 45


In [16]:
# Match each target point to a containing playground polygon.
playground_spatial_matches = gpd.sjoin(
    melbourne_points,
    named_playgrounds,
    how="left",
    predicate="within",
)

playground_candidates = (
    playground_spatial_matches.loc[
        playground_spatial_matches[
            "name_clean"
        ].notna()
    ]
    .copy()
)

print(
    "Targets inside a named playground polygon:",
    playground_candidates["place_id"].nunique(),
)

Targets inside a named playground polygon: 3


In [17]:
# Identify points contained by multiple playground polygons.
candidate_counts = (
    playground_candidates
    .groupby("place_id")
    .size()
)

multiple_match_count = (
    candidate_counts.gt(1).sum()
)

print(
    "Targets with multiple playground matches:",
    multiple_match_count,
)


# Summarise candidates by Vicmap activity category.
playground_match_summary = (
    playground_candidates
    .groupby("activity_category")
    .size()
    .rename("matched")
    .to_frame()
)

target_summary = (
    melbourne_points
    .groupby("activity_category")
    .size()
    .rename("target")
)

playground_match_summary = (
    target_summary.to_frame()
    .join(playground_match_summary)
    .fillna(0)
    .astype(int)
)

playground_match_summary["unmatched"] = (
    playground_match_summary["target"]
    - playground_match_summary["matched"]
)

display(playground_match_summary)

Targets with multiple playground matches: 0


,target,matched,unmatched
activity_category,,,
court,2,0,2
park_and_garden,5,0,5
playground,8,3,5
sports_ground,12,0,12


In [18]:
# Inspect every containment candidate before accepting it.
playground_review_columns = [
    "place_id",
    "activity_category",
    "feature_subtype",
    "display_name",
    "name_clean",
    "features",
    "council_re",
    "longitude",
    "latitude",
]

display(
    playground_candidates[
        playground_review_columns
    ].sort_values(
        ["activity_category", "name_clean"]
    )
)

,place_id,activity_category,feature_subtype,display_name,name_clean,features,council_re,longitude,latitude
352,vicmap_foi_1183091,playground,playground,Unnamed Playground - Melbourne - 1183091,Eades Park Playground,"Fitness track, slide, rockers, climbing featur...",1450812,144.951781,-37.807124
353,vicmap_foi_1183092,playground,playground,Unnamed Playground - Melbourne - 1183092,Eades Park Playground,"Fitness track, slide, rockers, climbing featur...",1450812,144.951765,-37.807256
355,vicmap_foi_985961,playground,playground,Unnamed Playground - Melbourne - 985961,Holland Park Playground,"Pirate ship play structure, rocking swing, roc...",1450808,144.925997,-37.797938


In [19]:
assert len(melbourne_points) == 27
assert melbourne_points["place_id"].is_unique
assert named_playgrounds["name_clean"].notna().all()

print("Playground polygon exploration completed.")

Playground polygon exploration completed.


### Nearest playground analysis

Five Melbourne playground targets were not contained within an official
playground polygon. Their distances to the nearest named City of Melbourne
playground are measured to determine whether the non-matches may result from
small coordinate or boundary differences.

Nearest features are retained only as exploratory candidates. Proximity alone
does not confirm that two records represent the same playground.

In [20]:
# Identify playgrounds without a strict polygon match.
strict_match_ids = set(
    playground_candidates["place_id"]
)

remaining_playgrounds = melbourne_points.loc[
    melbourne_points["activity_category"].eq(
        "playground"
    )
    & ~melbourne_points["place_id"].isin(
        strict_match_ids
    )
].copy()

print(
    "Playgrounds without containment match:",
    len(remaining_playgrounds),
)

Playgrounds without containment match: 5


In [21]:
# Project geometries so distances are measured in metres.
remaining_projected = remaining_playgrounds.to_crs(
    "EPSG:7855"
)

playground_polygons_projected = (
    named_playgrounds.to_crs("EPSG:7855")
)

nearest_playground_matches = gpd.sjoin_nearest(
    remaining_projected,
    playground_polygons_projected,
    how="left",
    distance_col="playground_distance_m",
)

print(
    "Nearest-match rows:",
    len(nearest_playground_matches),
)

Nearest-match rows: 5


In [22]:
# Check whether a target has multiple equally near polygons.
nearest_candidate_counts = (
    nearest_playground_matches
    .groupby("place_id")
    .size()
)

equal_distance_matches = (
    nearest_candidate_counts.gt(1).sum()
)

print(
    "Targets with multiple equally near playgrounds:",
    equal_distance_matches,
)

Targets with multiple equally near playgrounds: 0


In [23]:
# Keep one nearest candidate for each target.
nearest_playground_candidates = (
    nearest_playground_matches
    .sort_values(
        [
            "place_id",
            "playground_distance_m",
            "name_clean",
        ]
    )
    .drop_duplicates(subset="place_id")
    .copy()
)

assert len(nearest_playground_candidates) == 5

In [24]:
# Measure candidate coverage under several thresholds.
distance_thresholds = [10, 25, 50, 100]

playground_distance_summary = pd.DataFrame(
    {
        "maximum_distance_m": distance_thresholds,
        "matched_playgrounds": [
            nearest_playground_candidates[
                "playground_distance_m"
            ].le(distance).sum()
            for distance in distance_thresholds
        ],
    }
)

playground_distance_summary[
    "remaining_unmatched"
] = (
    len(nearest_playground_candidates)
    - playground_distance_summary[
        "matched_playgrounds"
    ]
)

display(playground_distance_summary)

,maximum_distance_m,matched_playgrounds,remaining_unmatched
0,10,1,4
1,25,1,4
2,50,1,4
3,100,1,4


In [25]:
# Inspect names and distances before choosing a threshold.
nearest_review_columns = [
    "place_id",
    "display_name",
    "name_clean",
    "playground_distance_m",
    "features",
    "council_re",
    "longitude",
    "latitude",
]

display(
    nearest_playground_candidates[
        nearest_review_columns
    ].sort_values("playground_distance_m")
)

,place_id,display_name,name_clean,playground_distance_m,features,council_re,longitude,latitude
349,vicmap_foi_1002117,Unnamed Playground - Melbourne - 1002117,Gardiner Reserve Playground,0.730503,"Multi-level play structure, climbers, see-saw,...",1450810,144.943859,-37.798876
350,vicmap_foi_1002130,Unnamed Playground - Melbourne - 1002130,North Melbourne Community Centre Playground,158.662334,"Combination units, swing, barbeque, some shade.",1450825,144.940681,-37.795022
356,vicmap_foi_985979,Unnamed Playground - Melbourne - 985979,North Melbourne Community Centre Playground,239.977692,"Combination units, swing, barbeque, some shade.",1450825,144.940323,-37.791305
354,vicmap_foi_1183141,Unnamed Playground - Melbourne - 1183141,Ievers Reserve Playground,284.199011,"Swings, rocker, fort, slide and climbing featu...",1450813,144.950764,-37.798674
351,vicmap_foi_1002133,Unnamed Playground - Melbourne - 1002133,Powlett Reserve Playground,587.869776,"Fort tower, sand pit, 4-way spring rocker, swi...",1450804,144.986310,-37.817008


In [26]:
assert len(strict_match_ids) == 3
assert len(remaining_playgrounds) == 5
assert nearest_playground_candidates[
    "place_id"
].is_unique

assert nearest_playground_candidates[
    "playground_distance_m"
].ge(0).all()

print("Nearest playground exploration completed.")

Nearest playground exploration completed.


In [29]:
# Create columns for manual review outcomes.
nearest_playground_candidates["review_status"] = "rejected"
nearest_playground_candidates["review_note"] = ""

# Accept the strong official match confirmed manually.
nearest_playground_candidates.loc[
    nearest_playground_candidates["place_id"].eq("vicmap_foi_1002117"),
    ["review_status", "review_note"],
] = [
    "accepted",
    "Official playground is 0.73 m away and was manually confirmed.",
]

# Record the possible Google Maps name without applying it.
nearest_playground_candidates.loc[
    nearest_playground_candidates["place_id"].eq("vicmap_foi_1002133"),
    ["review_status", "review_note"],
] = [
    "needs_official_confirmation",
    "Google Maps shows Yarra Park Playground, but Powlett Reserve is not a valid match.",
]

display(
    nearest_playground_candidates[
        [
            "place_id",
            "name_clean",
            "playground_distance_m",
            "review_status",
            "review_note",
        ]
    ]
)

,place_id,name_clean,playground_distance_m,review_status,review_note
349,vicmap_foi_1002117,Gardiner Reserve Playground,0.730503,accepted,Official playground is 0.73 m away and was man...
350,vicmap_foi_1002130,North Melbourne Community Centre Playground,158.662334,rejected,
351,vicmap_foi_1002133,Powlett Reserve Playground,587.869776,needs_official_confirmation,"Google Maps shows Yarra Park Playground, but P..."
354,vicmap_foi_1183141,Ievers Reserve Playground,284.199011,rejected,
356,vicmap_foi_985979,North Melbourne Community Centre Playground,239.977692,rejected,


### Nearest-playground review result

The five unmatched playground records were reviewed using distance and manual
map inspection. Only **Gardiner Reserve Playground**, located 0.73 metres from
the Vicmap point, was accepted.

The remaining four candidates could not be reliably linked to the target
locations. Google Maps was used only for visual checking, and its displayed
names were not adopted. These four records therefore retain their existing
generated names.

In [30]:
# Select the remaining non-playground targets.
landmark_target_categories = [
    "court",
    "park_and_garden",
    "sports_ground",
]

landmark_targets = melbourne_points.loc[
    melbourne_points["activity_category"].isin(
        landmark_target_categories
    )
].copy()

# Keep the two relevant official landmark sub-themes.
relevant_subthemes = [
    "Informal Outdoor Facility (Park/Garden/Reserve)",
    "Major Sports & Recreation Facility",
]

landmark_sources = recreation_landmarks.loc[
    recreation_landmarks["sub_theme"].isin(relevant_subthemes)
    & recreation_landmarks["feature_name_clean"].notna(),
    [
        "feature_name_clean",
        "theme",
        "sub_theme",
        "geometry",
    ],
].copy()

print("Remaining targets:", len(landmark_targets))
print("Relevant landmark records:", len(landmark_sources))

display(
    landmark_targets[
        [
            "place_id",
            "activity_category",
            "feature_subtype",
            "longitude",
            "latitude",
        ]
    ]
)

Remaining targets: 19
Relevant landmark records: 51


,place_id,activity_category,feature_subtype,longitude,latitude
78,vicmap_foi_1019251,court,basketball court,144.925151,-37.788034
79,vicmap_foi_1206820,court,netball court,144.947382,-37.784832
276,vicmap_foi_1000946,park_and_garden,park,144.954686,-37.824169
277,vicmap_foi_1002109,park_and_garden,park,144.947508,-37.819905
278,vicmap_foi_643094,park_and_garden,park,144.936174,-37.790994
279,vicmap_foi_643102,park_and_garden,park,144.935922,-37.792245
280,vicmap_foi_643110,park_and_garden,park,144.935558,-37.794182
468,vicmap_foi_1167974,sports_ground,sports ground,144.944033,-37.778165
469,vicmap_foi_1167979,sports_ground,sports ground,144.943773,-37.777745
470,vicmap_foi_1167990,sports_ground,sports ground,144.945422,-37.777922


In [31]:
# Project both datasets so distances are measured in metres.
landmark_targets_projected = landmark_targets.to_crs(
    "EPSG:7855"
)

landmark_sources_projected = landmark_sources.to_crs(
    "EPSG:7855"
)

# Find the nearest official landmark for each target.
nearest_landmark_matches = gpd.sjoin_nearest(
    landmark_targets_projected,
    landmark_sources_projected,
    how="left",
    distance_col="landmark_distance_m",
)

# Keep the nearest candidate for each target.
nearest_landmark_candidates = (
    nearest_landmark_matches
    .sort_values(
        [
            "place_id",
            "landmark_distance_m",
            "feature_name_clean",
        ]
    )
    .drop_duplicates(subset="place_id")
    .copy()
)

# Summarise coverage at several distance thresholds.
distance_thresholds = [10, 25, 50, 100, 200]

landmark_distance_summary = pd.DataFrame(
    {
        "maximum_distance_m": distance_thresholds,
        "matched_locations": [
            nearest_landmark_candidates[
                "landmark_distance_m"
            ].le(distance).sum()
            for distance in distance_thresholds
        ],
    }
)

landmark_distance_summary["unmatched_locations"] = (
    len(landmark_targets)
    - landmark_distance_summary["matched_locations"]
)

display(landmark_distance_summary)

,maximum_distance_m,matched_locations,unmatched_locations
0,10,0,19
1,25,0,19
2,50,0,19
3,100,2,17
4,200,5,14


In [32]:
# Inspect landmark candidates within 200 metres.
landmark_review = nearest_landmark_candidates.loc[
    nearest_landmark_candidates[
        "landmark_distance_m"
    ].le(200)
].copy()

review_columns = [
    "place_id",
    "activity_category",
    "feature_subtype",
    "display_name",
    "feature_name_clean",
    "sub_theme",
    "landmark_distance_m",
    "longitude",
    "latitude",
]

display(
    landmark_review[review_columns]
    .sort_values("landmark_distance_m")
)

,place_id,activity_category,feature_subtype,display_name,feature_name_clean,sub_theme,landmark_distance_m,longitude,latitude
476,vicmap_foi_1206821,sports_ground,sports ground,Unnamed Sports Ground - Melbourne - 1206821,State Netball Hockey Centre,Major Sports & Recreation Facility,63.364772,144.947473,-37.786350
477,vicmap_foi_1206848,sports_ground,sports ground,Unnamed Sports Ground - Melbourne - 1206848,State Netball Hockey Centre,Major Sports & Recreation Facility,84.943604,144.948449,-37.786119
79,vicmap_foi_1206820,court,netball court,Unnamed Netball Court - Melbourne - 1206820,State Netball Hockey Centre,Major Sports & Recreation Facility,107.334933,144.947382,-37.784832
277,vicmap_foi_1002109,park_and_garden,park,Unnamed Park - Melbourne - 1002109,Docklands Park,Informal Outdoor Facility (Park/Garden/Reserve),136.865741,144.947508,-37.819905
78,vicmap_foi_1019251,court,basketball court,Unnamed Basketball Court - Melbourne - 1019251,Newmarket Reserve,Informal Outdoor Facility (Park/Garden/Reserve),193.023501,144.925151,-37.788034


In [33]:
# Record the manual review decisions.
landmark_review["review_status"] = "rejected"
landmark_review["review_note"] = (
    "The official candidate could not be confirmed."
)

review_decisions = {
    "vicmap_foi_1206821": (
        "accepted",
        "Confirmed within the State Netball Hockey Centre complex.",
    ),
    "vicmap_foi_1206848": (
        "accepted",
        "Confirmed within the State Netball Hockey Centre complex.",
    ),
    "vicmap_foi_1206820": (
        "accepted",
        "Netball court confirmed within the State Netball Hockey Centre complex.",
    ),
    "vicmap_foi_1002109": (
        "accepted",
        "Target confirmed within the Docklands Park recreation area.",
    ),
    "vicmap_foi_1019251": (
        "rejected",
        "The target could not be reliably linked to Newmarket Reserve.",
    ),
}

# Apply each recorded decision.
for place_id, decision in review_decisions.items():
    status, note = decision
    mask = landmark_review["place_id"].eq(place_id)

    landmark_review.loc[
        mask,
        ["review_status", "review_note"],
    ] = [status, note]

display(
    landmark_review[
        [
            "place_id",
            "feature_name_clean",
            "landmark_distance_m",
            "review_status",
            "review_note",
        ]
    ].sort_values("landmark_distance_m")
)

,place_id,feature_name_clean,landmark_distance_m,review_status,review_note
476,vicmap_foi_1206821,State Netball Hockey Centre,63.364772,accepted,Confirmed within the State Netball Hockey Cent...
477,vicmap_foi_1206848,State Netball Hockey Centre,84.943604,accepted,Confirmed within the State Netball Hockey Cent...
79,vicmap_foi_1206820,State Netball Hockey Centre,107.334933,accepted,Netball court confirmed within the State Netba...
277,vicmap_foi_1002109,Docklands Park,136.865741,accepted,Target confirmed within the Docklands Park rec...
78,vicmap_foi_1019251,Newmarket Reserve,193.023501,rejected,The target could not be reliably linked to New...


### Landmark candidate review result

Five City of Melbourne landmark candidates were found within 200 metres and
manually reviewed.

Three Vicmap features were confirmed as being located within the larger
**State Netball Hockey Centre** complex. The official facility name was
accepted as a shared contextual name for these records. The park target within
the **Docklands Park** recreation area was also accepted.

The remaining candidate could not be reliably associated with **Newmarket
Reserve** and was rejected. Its existing generated name was retained.

Overall, four landmark names were accepted and one candidate was rejected.
Google Maps was used only for visual confirmation; no Google Maps names were
added to the dataset.

# City of Melbourne Name Enrichment Exploration

## Purpose

This exploration assessed whether official City of Melbourne datasets could
replace generated names for 27 unnamed Vicmap locations within the Melbourne
LGA.

## Data sources

Two official City of Melbourne datasets were explored:

- Playgrounds
- Landmarks and Places of Interest

The playground polygons were used for playground matching, while recreation
landmarks were used for courts, parks and sports grounds.

## Method

Vicmap locations were matched to official features using polygon containment
and nearest-distance matching. Candidates that were not direct spatial matches
were manually reviewed. Google Maps was used only for visual confirmation, and
no Google Maps names were added to the data.

## Results

Of the 27 target locations:

- 4 playground names were accepted.
- 4 landmark names were accepted.
- 19 locations retained their existing generated names.

A total of **8 official City of Melbourne names** were identified for
enrichment. The accepted records will be processed and validated in the next
pipeline stages before being applied to the final Vicmap dataset.